# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules to keep in mind 💡
1. Every bar chart **must have a zero baseline** — no exceptions
2. Every chart **must have an insight title**, not a topic title
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names

---


## Setup — Run this cell first


In [2]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
df.head()


Dataset: 63 countries, 9 columns


,Country,Region,Happiness_Score,GDP,Social_Support,Life_Expectancy,Freedom,Generosity,Corruption
0,Finland,Western Europe,7.804,10.775,0.954,71.9,0.949,0.142,0.179
1,Denmark,Western Europe,7.586,10.933,0.954,72.7,0.931,0.168,0.234
2,Iceland,Western Europe,7.525,10.878,0.983,72.5,0.961,0.260,0.150
3,Israel,Middle East and North Africa,7.473,10.527,0.916,72.4,0.903,0.149,0.826
4,Netherlands,Western Europe,7.464,11.015,0.939,72.4,0.879,0.240,0.296


In [3]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [ ]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # ascending: Plotly draws the first row at the bottom

print(region_avg)

# Step 2: Build your chart
top_region = region_avg.iloc[-1]
bottom_region = region_avg.iloc[0]
ratio = top_region['Happiness_Score'] / bottom_region['Happiness_Score']

# Colour: highlight the happiest and least happy regions, grey out the rest
HIGHLIGHT_TOP, HIGHLIGHT_BOTTOM, NEUTRAL = '#1b7f79', '#d1603d', '#c9ced6'
colors = [NEUTRAL] * len(region_avg)
colors[0] = HIGHLIGHT_BOTTOM
colors[-1] = HIGHLIGHT_TOP

fig1 = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker_color=colors,
    text=region_avg['Happiness_Score'].round(2),
    textposition='outside',
    cliponaxis=False,
    hovertemplate='%{y}: %{x:.2f}<extra></extra>',
))

# Global average reference line
fig1.add_vline(x=df['Happiness_Score'].mean(), line_dash='dash', line_color='#555',
               annotation_text=f"Global avg {df['Happiness_Score'].mean():.2f}",
               annotation_position='bottom right', annotation_font_size=12)

fig1.update_layout(
    title=dict(
        text=(f"<b>{top_region['Region']} is {ratio:.1f}x as happy as {bottom_region['Region']}</b><br>"
              "<sup>Average happiness score by region, World Happiness Report 2023 (scale 0-10)</sup>"),
        x=0.0, xanchor='left'),
    xaxis=dict(range=[0, 8.5], title='Average happiness score', showgrid=True,
               gridcolor='#eceff3', zeroline=True, zerolinecolor='#333'),
    yaxis=dict(title='', tickfont=dict(size=13)),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Helvetica, Arial, sans-serif', size=13, color='#222'),
    showlegend=False,
    margin=dict(l=20, r=40, t=100, b=60),
    height=500, width=900,
)
fig1.show()

---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [ ]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build your chart
TOP_COLOR, BOTTOM_COLOR = '#1b7f79', '#d1603d'
bar_colors = combined['Group'].map({'Top 8': TOP_COLOR, 'Bottom 8': BOTTOM_COLOR})

gap = top8['Happiness_Score'].min() - bottom8['Happiness_Score'].max()
avg_top, avg_bottom = top8['Happiness_Score'].mean(), bottom8['Happiness_Score'].mean()

fig2 = go.Figure(go.Bar(
    x=combined['Happiness_Score'],
    y=combined['Country'],
    orientation='h',
    marker_color=bar_colors,
    text=combined['Happiness_Score'].round(2),
    textposition='outside',
    cliponaxis=False,
    hovertemplate='%{y}: %{x:.2f}<extra></extra>',
))

# Visual separator between the two groups (between bar 8 and bar 9, counting from 0)
fig2.add_hline(y=7.5, line_color='#888', line_width=2, line_dash='dot')

# Stretch goal: global average reference line
fig2.add_vline(x=global_avg, line_dash='dash', line_color='#555',
               annotation_text=f"Global avg {global_avg:.2f}", annotation_position='top',
               annotation_font_size=12)

# Gap annotation on the separator
fig2.add_annotation(
    x=8.6, y=7.5, xref='x', yref='y', showarrow=False, xanchor='right',
    text=f"<b>{gap:.1f}-point gap</b><br>between the groups",
    font=dict(size=13, color='#222'), bgcolor='white', bordercolor='#888', borderwidth=1, borderpad=4,
)

fig2.update_layout(
    title=dict(
        text=(f"<b>The happiest countries score {avg_top / avg_bottom:.1f}x higher than the least happy</b><br>"
              "<sup>Top 8 vs. bottom 8 countries, World Happiness Report 2023 (scale 0-10)</sup>"),
        x=0.0, xanchor='left'),
    xaxis=dict(range=[0, 9], title='Happiness score', showgrid=True, gridcolor='#eceff3',
               zeroline=True, zerolinecolor='#333'),
    yaxis=dict(title='', categoryorder='array', categoryarray=list(combined['Country']),
               tickfont=dict(size=12)),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Helvetica, Arial, sans-serif', size=13, color='#222'),
    showlegend=False,
    margin=dict(l=20, r=40, t=100, b=60),
    height=700, width=900,
)

# Colour-coded group labels instead of a legend
fig2.add_annotation(x=0.99, y=0.97, xref='paper', yref='paper', showarrow=False, xanchor='right',
                    text='<b>Top 8</b>', font=dict(color=TOP_COLOR, size=14))
fig2.add_annotation(x=0.99, y=0.03, xref='paper', yref='paper', showarrow=False, xanchor='right',
                    text='<b>Bottom 8</b>', font=dict(color=BOTTOM_COLOR, size=14))
fig2.show()

---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [ ]:
# Stretch goal — grouped bar chart
# --------------------------------
# GDP is on a log scale (~7-12) while Freedom is 0-1, so they can't share an axis directly.
# Fix: rescale both to a 0-100 index (0 = lowest country in the dataset, 100 = highest).
# Note: the region in this dataset is called 'Latin America and Caribbean'.

regions = ['Western Europe', 'Latin America and Caribbean', 'East Asia',
           'Sub-Saharan Africa', 'South Asia']
factors = {'GDP': 'GDP per capita', 'Freedom': 'Freedom'}

scaled = df.copy()
for col in factors:
    scaled[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min()) * 100

grouped = (scaled[scaled['Region'].isin(regions)]
           .groupby('Region')[list(factors)].mean()
           .reindex(regions))
print(grouped.round(1))

rich, free = grouped['GDP'].idxmax(), grouped['Freedom'].idxmax()
title3 = (f"<b>Wealth isn't freedom: {rich} leads on GDP, but {free} leads on freedom</b>"
          if rich != free else f"<b>{rich} leads on both wealth and freedom</b>")

fig3 = go.Figure()
for col, color in zip(factors, ['#2f5d8a', '#e0a526']):
    fig3.add_trace(go.Bar(
        name=factors[col], x=grouped.index, y=grouped[col], marker_color=color,
        text=grouped[col].round(0).astype(int), textposition='outside',
        hovertemplate='%{x}<br>' + factors[col] + ': %{y:.0f}<extra></extra>',
    ))

fig3.update_layout(
    barmode='group',
    title=dict(
        text=(title3 + "<br>"
              "<sup>Average GDP per capita and Freedom by region, rescaled to a 0-100 index</sup>"),
        x=0.0, xanchor='left'),
    yaxis=dict(range=[0, 105], title='Index (0 = lowest, 100 = highest country)',
               showgrid=True, gridcolor='#eceff3', zeroline=True, zerolinecolor='#333'),
    xaxis=dict(title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Helvetica, Arial, sans-serif', size=13, color='#222'),
    legend=dict(orientation='h', y=1.02, x=1, xanchor='right', yanchor='bottom'),
    margin=dict(l=20, r=40, t=110, b=60),
    height=550, width=950,
)
fig3.show()